In [ ]:
%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2

In [ ]:
import base64
import os
import re
import sys
import uuid
from io import BytesIO
from typing import Tuple

sys.path.insert(0, os.path.abspath(".."))

import wikipedia
from agents import Agent, Runner, function_tool
from agents.extensions.memory import RedisSession
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings
from neo4j_graphrag.retrievers import HybridRetriever
from neo4j_graphrag.types import LLMMessage
from PIL import Image

from core import chat
from core.models import Settings
from core.utils import record_to_dict

settings = Settings()

In [ ]:
@function_tool
def wikipedia_lookup(query: str, limit: int = 3) -> list[dict[str, str]] | str:
    """
    Search Wikipedia and return structured summaries of the top matching pages.

    This tool performs a Wikipedia search using the provided query and retrieves
    up to `limit` matching page titles. For each result, it attempts to fetch
    the full page summary and returns structured Python objects suitable for
    downstream agent reasoning.

    Behavior:
    - Executes `wikipedia.search(query, results=limit)` to retrieve page titles.
    - For each page title:
        - If successful, returns:
            {
              "document": "<page title>",
              "summary": "<full page summary>"
            }
        - If a disambiguation page is encountered, returns:
            {
              "document": "<page title>",
              "error": "Disambiguation error - [list of options]"
            }
        - If the page cannot be found, returns:
            {
              "document": "<page title>",
              "error": "Page not found."
            }

    - If no search results are found, returns:
        "No results found on Wikipedia."

    - If the initial query itself raises a disambiguation error, returns:
        "Disambiguation error - [list of options]"

    - If the page does not exist at all, returns:
        "Page not found."

    Args:
        query (str):
            The search term to look up on Wikipedia. This can represent a person,
            historical event, location, Roman emperor, coin type, mint, dynasty,
            or any Roman history or numismatic topic.

        limit (int, optional):
            The maximum number of search results to retrieve and summarize.
            Default is 3.

    Returns:
        list[dict[str, str]] | str:
            On success:
                A list of dictionaries, where each dictionary contains:
                    - "document": The Wikipedia page title
                    - "summary": The page summary (if available)
                      OR
                    - "error": A description of the issue encountered

            On failure:
                A plain error string.

    Agent Usage Guidelines:
        - Use this tool for general Roman history or numismatics questions.
        - Use it to enrich answers with historical background after identifying
          a coin, emperor, mint, or historical event.
        - If multiple documents are returned:
            * Prefer entries with valid summaries over those containing errors.
            * Select the most relevant entry based on the user’s question.
        - Do not fabricate historical details beyond what is supported by
          Wikipedia and model knowledge.

    Notes:
        - Returns structured Python objects (not JSON).
        - Summary length depends on Wikipedia's default behavior.
        - Designed for AI tool usage and structured reasoning workflows.
    """
    try:
        documents = wikipedia.search(query, results=limit)
        if not documents:
            return "No results found on Wikipedia."

        summaries = []
        for doc in documents:
            try:
                summary = wikipedia.summary(doc)
                summaries.append({"document": doc, "summary": summary})
            except wikipedia.exceptions.DisambiguationError as e:
                summaries.append(
                    {"document": doc, "error": f"Disambiguation error - {e.options}"}
                )
            except wikipedia.exceptions.PageError:
                summaries.append({"document": doc, "error": "Page not found."})
        return summaries
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Disambiguation error - {e.options}"
    except wikipedia.exceptions.PageError:
        return "Page not found."

In [ ]:
@function_tool
def coin_db_lookup(query: str, max_results: int = 10) -> list[dict[str, str]] | str:
    """
    Search the Roman coin database using hybrid semantic and full-text retrieval.

    This tool queries a Neo4j-backed Roman coin database using a hybrid
    retrieval strategy that combines:

        - Vector similarity search (OpenAI embeddings)
        - Full-text keyword search

    It is optimized for identifying Roman Imperial and Late Roman coins,
    particularly RIC-based corpus entries.

    The tool is suitable for queries containing:
        - Emperor names
        - Partial or complete obverse/reverse legends
        - Reverse type descriptions
        - Mint names
        - Date ranges
        - Exact or partial RIC references
        - Natural language coin descriptions

    Args:
        query (str):
            A descriptive search string representing a Roman coin.
            May include:
                - Free-text descriptions
                - Obverse or reverse legends
                - RIC numbers (e.g., "RIC VII Rome 66")
                - Mint names
                - Emperor names
                - Date information

            Example inputs:
                "Constantine I quadriga reverse 336"
                "RIC VII Constantinople 103"
                "MONETA AVGG three Monetae Rome"
                "D N CONSTANTINVS MAX AVG radiate bust"

        max_results (int, optional):
            Maximum number of top-ranked results to return.
            Default is 10.

    Returns:
        list[dict[str, str]] | str:

        On success:
            A list of structured coin records. Each record contains:

                {
                  "record_id": str,     # Unique identifier (RIC-based key)
                  "title": str,         # Canonical title (e.g., "RIC VII Rome 66")
                  "date_range": str,    # Human-readable date range (may be empty)
                  "from_year": int,     # Start year (AD)
                  "to_year": int,       # End year (AD)
                  "object_type": str,   # e.g., "Coin", "Medallion"
                  "description": str    # Obverse and reverse description
                }

        On failure:
            A plain error string describing the database issue.

    Agent Usage Guidelines:
        - Use this tool AFTER:
            * A coin has been identified from image analysis, OR
            * The user asks to identify a coin from a textual description.

        - Construct a concise but information-rich query including:
            * Emperor name (if known)
            * Key legend fragments
            * Reverse type description
            * Mint (if known)
            * Date range (if known)
            * RIC reference (if mentioned)

        - If multiple results are returned:
            1. Prioritize exact RIC matches (if provided).
            2. Then prioritize legend matches.
            3. Then match reverse type and mint.
            4. Use date range as secondary validation.
            5. Select the closest overall match.

        - If no results are returned:
            * Inform the user the coin was not found in the database.
            * Provide a best-effort identification using available evidence.
            * Do NOT fabricate RIC numbers or metadata.

        - Never invent database records or fields not present in results.

    Notes:
        - Results are ranked using hybrid semantic + keyword relevance.
        - Returns structured Python objects (not JSON).
        - Designed for deterministic AI-assisted coin identification workflows.
    """
    VECTOR_INDEX = "coins_embeddings_index"
    FULL_TEXT_INDEX = "coins_text_index"
    try:
        driver = GraphDatabase.driver(
            settings.neo4j_uri,
            auth=(settings.neo4j_username, settings.neo4j_password),
        )
        embedder = OpenAIEmbeddings(
            model=settings.embedding_model, api_key=settings.openai_api_key
        )
        retriever = HybridRetriever(
            driver,
            VECTOR_INDEX,
            FULL_TEXT_INDEX,
            embedder,
            neo4j_database=settings.neo4j_database,
        )
        results = retriever.get_search_results(query, top_k=max_results)
        data = []
        for record in results:
            data.extend(
                [
                    record_to_dict(r, selector="node")
                    for r in record[1]
                    if not isinstance(r, str)
                ]
            )

        driver.close()
        return data
    except Exception as e:
        return f"Error querying coin database: {str(e)}"

In [ ]:
SAFE_IMAGE_DIR = os.path.abspath("../data/coins")
FILENAME_RE = re.compile(r"^[A-Za-z0-9._-]+\.(jpg|jpeg|png)$", re.IGNORECASE)


def is_valid_image(file_path: str) -> Tuple[bool, str]:
    try:
        with Image.open(file_path) as img:
            if img.format in ("PNG", "JPEG"):
                return True, Image.MIME.get(img.format)
            else:
                return False, ""
    except Exception:
        return False, ""


def resize_for_model(filename: str, max_dim: int = 768) -> bytes:
    with Image.open(filename) as image:
        if image.width > max_dim or image.height > max_dim:
            image.thumbnail((max_dim, max_dim), Image.LANCZOS)
        buf = BytesIO()
        save_format = "PNG" if image.format == "PNG" else "JPEG"
        image.save(buf, format=save_format)
        return buf.getvalue()


@function_tool
def read_coin_image_as_data_url(filename: str) -> dict[str, str]:
    """
    Securely load a local coin image file (JPEG or PNG) and return it
    as a structured image input item for vision-capable models.

    This tool reads an image from a restricted server directory and
    converts it into a Base64-encoded data URL suitable for OpenAI
    multimodal input.

    Security Guardrails:
        - The filename must be a plain filename (no directory paths).
        - Only .jpg, .jpeg, or .png extensions are allowed.
        - Absolute paths, relative paths, and path traversal (e.g., "..")
          are strictly rejected.
        - The file must exist inside the configured SAFE_IMAGE_DIR.
        - The file must be validated as a real JPEG or PNG image.
        - Non-image files or corrupted images are rejected.

    Args:
        filename (str):
            A simple filename such as:
                "coin.jpg"
                "denarius.png"
            The value must NOT contain path separators or directory references.

    Returns:
        dict[str, str]:
            A structured image input item formatted for OpenAI Agents SDK:

                {
                  "type": "input_image",
                  "image_url": "data:image/jpeg;base64,..."
                }

            The MIME type will be either:
                - image/jpeg
                - image/png

    Agent Usage Guidelines:
        - Use this tool when the user requests to identify or analyze
          a coin image file stored on the server.
        - After receiving the returned image item, pass it directly to
          the model for visual analysis.
        - If the tool raises an error, inform the user that:
              * Only JPEG or PNG files are supported
              * The filename must not include paths
              * The file must exist in the allowed directory
        - Do NOT attempt to manually construct data URLs.
        - Do NOT bypass this tool for image loading.

    Notes:
        - Designed for secure image access.
        - Prevents directory traversal attacks.
        - Ensures only validated image files are processed.
        - Returns structured data (not plain text).
    """
    if not filename or len(filename) > 120:
        raise ValueError("Invalid filename length.")

    if "/" in filename or "\\" in filename or ".." in filename:
        raise ValueError(
            "Invalid filename. Use a plain JPEG or PNG filename like 'coin.jpg' or 'coin.png' (no paths, no '..')."
        )

    if not FILENAME_RE.match(filename):
        raise ValueError(
            "File must be a .jpg/.jpeg/.png filename (e.g., 'coin.jpg' or 'coin.png')."
        )

    base_dir = os.path.abspath(SAFE_IMAGE_DIR)
    abs_path = os.path.abspath(os.path.join(base_dir, filename))

    # Ensure it cannot escape SAFE_IMAGE_DIR
    if not abs_path.startswith(base_dir + os.sep):
        raise ValueError("Invalid filename (path escape detected).")

    # Verify the file exists and is a valid image
    if not os.path.isfile(abs_path):
        raise ValueError("File does not exist.")

    valid, mime_type = is_valid_image(abs_path)
    if not valid:
        raise ValueError("File is not a valid JPEG or PNG image.")

    data = resize_for_model(abs_path, max_dim=768)
    b64 = base64.b64encode(data).decode("ascii")
    data_url = f"data:{mime_type};base64,{b64}"

    # IMPORTANT: return as an image item so the agent can pass it to the model as vision input
    return {"type": "input_image", "image_url": data_url}

In [ ]:
ROMAN_NUMISMATICS_INSTRUCTIONS = """
You are a specialist assistant for Roman history and Roman numismatics.

ROUTING RULES:

A) If the user asks to read/identify a coin image file:
  1) Call read_coin_image_as_data_url(filename=...) with the provided filename.
     - If the tool rejects it, explain that ONLY a plain .jpg/.jpeg filename is allowed (no paths).
  2) After receiving the image item, decide whether it is a Roman coin.
     - If NOT a Roman or a Byzantine coin: reply exactly:
       "This object cannot be recognized as a Roman or Byzantine coin."
       Additionally, explain what you see on the picture (describe the visible features, text, or objects).
       Then STOP.
     - If it IS a Roman or Byzantine coin:
       Extract all visible metadata (best effort):
         * obverse legend (as seen)
         * reverse legend (as seen)
         * bust/type description
         * reverse type description
         * denomination (best effort)
         * mint and date range (best effort)
         * emperor/issuer (best effort)
         * If possible: OCRE identifier and/or RIC code
           - Never invent these. If uncertain, say you are unsure.
  3) Then call coin_db_lookup using the extracted fields (include ric/ocreId if available).
     - If DB returns match(es): answer using DB info + your extracted observations.
     - If DB returns not found: say it's not found in the database, then provide a generic best-effort answer
       using your extracted observations.

B) If the user asks a general Roman history or numismatics question (no file/image):
  - Use wikipedia_lookup for key topic(s) and answer using Wikipedia plus your own knowledge.

C) Otherwise:
  - Say you only answer questions about Roman history and numismatics.
"""


class RomanNumismaticsAgent(chat.ChatService):
    def __init__(self):
        super().__init__()

        self.session_id = uuid.uuid4().hex
        self.session = RedisSession.from_url(self.session_id, url=settings.redis_url)
        self.agent = Agent(
            name="Roman Numismatics Assistant",
            instructions=ROMAN_NUMISMATICS_INSTRUCTIONS,
            tools=[read_coin_image_as_data_url, coin_db_lookup, wikipedia_lookup],
            model=settings.chat_model,
        )

    async def chat(self, query: str, history: list[LLMMessage]) -> str:
        result = await Runner.run(self.agent, query, session=self.session)
        return result.final_output

    def clear(self):
        self.session.clear_session()


chat.run(RomanNumismaticsAgent())